# RNA Seq Pipeline
#### Author: Kylie Gregory 
#### Created On: 11/25/2026
#### Last Modified: 11/25/2026
#### Purpose: Create an RNA-seq pipeline to determine what gene expression changes are consistent across two independent datasets comparing dilated cardiomyopathy (DCM) and healthy (non-failing donor) groups. 

#### Files: 


In [3]:
#load in libraries

library(DESeq2)
library(tidyverse)
library(GEOquery)

In [4]:
#read in RNA-seq raw counts dataset


create_raw_counts_table <- function(raw_counts_url, GEO_number){
    #either GSE141910.tsv.gz or GSE116250.tsv.gz
    file_name <- paste(GEO_number, ".tsv.gz", sep="")
    download.file(raw_counts_url, file_name)
    count_table <- read.table(file_name, header=T,sep="\t")
    #make GeneID column the unique row identifiers 
    count_table <- column_to_rownames(count_table, var ="GeneID")

    return(count_table)
    
}

In [5]:

#load in column data to map each sample to a condition 

create_column_data <- function(GEO_number){
                               
    #get the GEO series                          
    gse <- getGEO(GEO_number)
                               
    #get the phenotypic data about each sample                           
    sample_data <- pData(gse[[1]])
    
     #the condition (DCM vs non-failing) is in different columns for the two datasets we are analysing, so created an if/else block 
                               
    #this bloc filters to only DCM vs non-failing samples, selects only that column, changes the name of the column to condition, and edits the exact wording of non-failing and DCM to be consistent among the datasets to allow downstream code to be modular
                              
                 
    if (GEO_number =="GSE141910")
        {
        col_data <- sample_data %>% filter(characteristics_ch1 %in% c("etiology: Non-Failing Donor",
                                                                    "etiology: Dilated cardiomyopathy (DCM)")) %>% select(characteristics_ch1) %>% rename(condition=characteristics_ch1)
        
        col_data$condition <- sub("etiology: ", "", col_data$condition)
            
    } else{
        col_data <-  col_data <- sample_data %>% filter(`disease:ch1` %in% c("non-failing", "dilated cardiomyopathy")) %>% select(`disease:ch1`) %>% rename(condition=`disease:ch1`)
        
        col_data$condition <- sub("non-failing", "Non-Failing Donor", col_data$condition)
        col_data$condition <- sub("dilated cardiomyopathy", "Dilated cardiomyopathy (DCM)", 
                                  col_data$condition)
    
    }
    
    
    return (col_data)

}



#Link for working with GEOquery: https://bioconductor.org/packages/release/bioc/vignettes/GEOquery/inst/doc/GEOquery.html#why-geoquery
                               



In [6]:
d1_geo_number <- "GSE141910"
d2_geo_number <- "GSE116250"

d1_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE141910&format=file&file=GSE141910_raw_counts_GRCh38.p13_NCBI.tsv.gz"
d2_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE116250&format=file&file=GSE116250_raw_counts_GRCh38.p13_NCBI.tsv.gz"

#get each datasets raw counts matrix
d1_count_table <- create_raw_counts_table(d1_count_url, d1_geo_number)
d2_count_table <- create_raw_counts_table(d2_count_url, d2_geo_number)

#get each datasets condition information 
d1_col_data <- create_column_data(d1_geo_number)
d2_col_data <- create_column_data(d2_geo_number)
    

Found 1 file(s)



GSE141910_series_matrix.txt.gz



Found 1 file(s)



GSE116250_series_matrix.txt.gz



Using locally cached version of GPL16791 found here:
/tmp/RtmpA0ibgi/GPL16791.soft.gz 



In [7]:
#count matrix has all of the samples, so need to filter to just get DCM and non-failing donor 
#for DESeq2, need rows of col_data to match columns of count_tale


create_filtered_tables <- function(col_data, count_table){
    
    #col_data contains only sample numbers belonging to DCM and non failing donors 
    col_data_samples <- rownames(col_data)
    
    #count_table contains sample numbers from every condition 
    all_samples <- colnames(count_table)
    
    #get overlapping sample numbers 
    dcm_nf_samples <- intersect(col_data_samples, all_samples)
    
    #new count table contains only DCM and nonfailing donors that we also have informatation about in col_data
    count_table_filtered <- count_table %>%
    select(all_of(dcm_nf_samples))
    
    #new col_data contains only DCM and nonfailing donors that
    #were also available in the filtered count table
    
    col_data_filtered <- col_data %>% filter(rownames(col_data) %in% dcm_nf_samples)
    
    tables <- list(count_table_filtered, col_data_filtered)
    
    return (tables)
    
}



In [8]:
d1_final_tables <- create_filtered_tables(d1_col_data, d1_count_table)
d2_final_tables <- create_filtered_tables(d2_col_data, d2_count_table)



d1_final_count_table <- d1_final_tables[[1]]
d2_final_count_table <- d2_final_tables[[1]]


d1_final_col_data  <- d1_final_tables[[2]]
d2_final_col_data  <- d2_final_tables[[2]]


In [9]:
#run DESeq analysis 


run_DESeq <- function(count_table, col_data){
    
    dds <- DESeqDataSetFromMatrix(count_table, col_data, design = ~ condition)
    dds <- DESeq(dds)
    
    return (dds) 
}



In [21]:
#find differentially expressed genes (p_adjusted < 0.05 and log2fold change >=1)


get_DESeq_results <- function(dds){
    
    res <- results(dds,contrast=c("condition","Dilated cardiomyopathy (DCM)", "Non-Failing Donor"), alpha = 0.05, lfcThreshold =1)
    
    res <- as_tibble(res)
    
    all_genes <- res %>% rownames_to_column(var = "GeneID")
    significant_genes <- res %>% filter(!is.na(res$padj) & padj < 0.05)
    upreg_genes <- significant_genes %>% filter(log2FoldChange > 0) %>% rownames_to_column(var = "GeneID")
    downreg_genes <- significant_genes %>% filter(log2FoldChange < 0) %>% rownames_to_column(var = "GeneID")
    
    genes <- list(upreg_genes, downreg_genes, all_genes)
    
    return (genes)
   
    
}


In [11]:
d1_dds <- run_DESeq(d1_final_count_table, d1_final_col_data)

d2_dds <- run_DESeq(d2_final_count_table, d2_final_col_data)

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating size factors



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating dispersions



gene-wise dispersion estimates



... (20 outputs left out)

-- replacing outliers and refitting for 186 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)



estimating dispersions



fitting model and testing



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



In [22]:
d1_genes_of_interest <- get_DESeq_results(d1_dds)
d1_up_genes <- d1_genes_of_interest[[1]]
d1_down_genes <- d1_genes_of_interest[[2]]
d1_all_genes <- d1_genes_of_interest[[3]]


d2_genes_of_interest <- get_DESeq_results(d2_dds)
d2_up_genes <- d2_genes_of_interest[[1]]
d2_down_genes <- d2_genes_of_interest[[2]]
d2_all_genes <- d2_genes_of_interest[[3]]

[1] 94  7

In [ ]:
#Get gene annotations to map gene IDs to their gene symbols 

create_gene_annot_table <- function(gene_annot_file)
    
    {
    
    gene_annot_table <- read.table(gene_annot_file, header=TRUE,
                                   sep = "\t", fill=TRUE, quote="")
    
    gene_annot_table_filtered <- gene_annot_table %>% select(GeneID, Symbol)
    
    return (gene_annot_table_filtered)
    
}

In [ ]:
gene_annot_file <- "Human.GRCh38.p13.annot.tsv.gz"
gene_annot_table <- create_gene_annot_table(gene_annot_file)

In [ ]:
#running fisher's exact test 
#documentation for test: https://bioconductor.org/packages/devel/bioc/manuals/GeneOverlap/man/GeneOverlap.pdf

# fisher.test(matrix(c(110, 20, 60, 10), nrow=2), alternative="greater")

# matrix(c(n - union(A,B), setdiff(A,B), setdiff(B,A), intersect(A,B)), nrow=2)

# n = # of genes, A = # genes in set A, B = # of genes in set B

#n - union(A,B) = significant in neither




